# OPF fine-tuning preparation for AymurAI NER train candidates

In [ ]:
from __future__ import annotations

import json
import os
import random
import shlex
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

In [ ]:
def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "aymurai").exists():
            return path
    return start


def validate_opf_checkpoint(checkpoint_dir: Path) -> tuple[bool, str]:
    if not checkpoint_dir.exists() or not checkpoint_dir.is_dir():
        return False, f"Checkpoint directory not found: {checkpoint_dir}"

    config_path = checkpoint_dir / "config.json"
    if not config_path.exists():
        return False, f"Missing config.json in checkpoint: {config_path}"

    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception as exc:
        return False, f"Invalid JSON in checkpoint config {config_path}: {exc}"

    if not isinstance(config.get("encoding"), str) or not config.get("encoding", "").strip():
        return (
            False,
            "Checkpoint config field `encoding` is missing/empty. "
            "This usually means you are pointing to a non-OPF-compatible folder. "
            "Try the `original/` subfolder of the HF repo snapshot.",
        )

    safetensors_files = sorted(checkpoint_dir.glob("*.safetensors"))
    if not safetensors_files:
        return False, f"No .safetensors files found in checkpoint: {checkpoint_dir}"

    # Detect unresolved Git LFS pointer files.
    for st_path in safetensors_files:
        try:
            head = st_path.read_text(encoding="utf-8", errors="ignore")[:128]
        except Exception:
            head = ""
        if "git-lfs.github.com/spec/v1" in head or st_path.stat().st_size < 1024 * 1024:
            return (
                False,
                "Checkpoint weights look like unresolved Git LFS pointers. "
                "Run `git lfs pull` inside your checkpoint repo.",
            )

    return True, "ok"


def resolve_opf_checkpoint(project_root: Path) -> tuple[Path | None, str | None]:
    env_value = os.environ.get("OPF_CHECKPOINT")
    candidates: list[Path] = []
    if env_value:
        env_path = Path(env_value).expanduser()
        candidates.append(env_path)
        candidates.append(env_path / "original")

    candidates.extend(
        [
            Path.home() / ".opf" / "privacy_filter" / "original",
            Path.home() / ".opf" / "privacy_filter",
            project_root / ".opf" / "privacy_filter" / "original",
            project_root / ".opf" / "privacy_filter",
            project_root / "resources" / "models" / "privacy_filter" / "original",
            project_root / "resources" / "models" / "privacy_filter",
            project_root / "models" / "privacy_filter" / "original",
            project_root / "models" / "privacy_filter",
        ]
    )

    seen: set[str] = set()
    last_error: str | None = None
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        valid, reason = validate_opf_checkpoint(candidate)
        if valid:
            return candidate, None
        if candidate.exists():
            last_error = f"{candidate}: {reason}"

    return None, last_error


PROJECT_ROOT = find_project_root(Path.cwd())

INPUT_JSONL = Path(
    os.environ.get(
        "AYMURAI_TRAIN_CANDIDATES_JSONL",
        str(
            PROJECT_ROOT
            / "resources"
            / "data"
            / "public-data"
            / "saij_cloud"
            / "train_candidates.jsonl"
        ),
    )
).expanduser()

OUTPUT_DIR = Path(
    os.environ.get(
        "OPF_OUTPUT_DIR",
        str(PROJECT_ROOT / "resources" / "outputs" / "opf-finetuning" / "aymurai_ner_v1"),
    )
).expanduser()

SEED = int(os.environ.get("OPF_SPLIT_SEED", "1337"))
TRAIN_RATIO = 0.8
VALIDATION_RATIO = 0.1
TEST_RATIO = 0.1
CATEGORY_VERSION = "aymurai_ner_v1"

RUN_OPF = os.environ.get("RUN_OPF", "1") == "1"
OPF_CHECKPOINT, OPF_CHECKPOINT_ERROR = resolve_opf_checkpoint(PROJECT_ROOT)
DEVICE = os.environ.get(
    "OPF_DEVICE",
    "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"),
)
EPOCHS = int(os.environ.get("OPF_EPOCHS", "3"))
BATCH_SIZE = int(os.environ.get("OPF_BATCH_SIZE", "1"))
LEARNING_RATE = float(os.environ.get("OPF_LEARNING_RATE", "1e-5"))

# Triton MoE kernels are optional and typically unavailable on Mac/MPS/CPU setups.
if os.environ.get("OPF_MOE_TRITON") is None and DEVICE != "cuda":
    os.environ["OPF_MOE_TRITON"] = "0"

if RUN_OPF and OPF_CHECKPOINT is None:
    RUN_OPF = False
    print(
        "Warning: OPF checkpoint directory is missing or invalid. "
        "Set OPF_CHECKPOINT to a valid local checkpoint path and re-run this cell."
    )
    if OPF_CHECKPOINT_ERROR:
        print(f"Checkpoint detail: {OPF_CHECKPOINT_ERROR}")
        print("Hint: if cloned from Hugging Face, run `git lfs pull` in that repo.")

TRAIN_JSONL = OUTPUT_DIR / "train.jsonl"
VALIDATION_JSONL = OUTPUT_DIR / "validation.jsonl"
TEST_JSONL = OUTPUT_DIR / "test.jsonl"
LABEL_SPACE_JSON = OUTPUT_DIR / "label_space.json"
REPORT_JSON = OUTPUT_DIR / "conversion_report.json"
FINETUNED_CHECKPOINT_DIR = OUTPUT_DIR / "finetuned_checkpoint"

assert abs((TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO) - 1.0) < 1e-9

print(f"Project root: {PROJECT_ROOT}")
print(f"Input JSONL: {INPUT_JSONL}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"RUN_OPF: {RUN_OPF}")
print(f"OPF_CHECKPOINT: {OPF_CHECKPOINT}")
print(f"DEVICE: {DEVICE}")
print(f"OPF_MOE_TRITON: {os.environ.get('OPF_MOE_TRITON')}")

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            payload = json.loads(line)
            if not isinstance(payload, dict):
                raise ValueError(
                    f"Expected JSON object at line {line_number}, got {type(payload).__name__}"
                )
            rows.append(payload)
    return rows


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


def label_frequency(
    rows: list[dict[str, Any]], field: str = "final_entities"
) -> Counter[str]:
    counts: Counter[str] = Counter()
    for row in rows:
        entities = row.get(field) or []
        if not isinstance(entities, list):
            continue
        for entity in entities:
            if not isinstance(entity, dict):
                continue
            label = str(entity.get("label") or "").strip()
            if label:
                counts[label] += 1
    return counts


def show_table(rows: list[dict[str, Any]], limit: int | None = None) -> None:
    visible = rows if limit is None else rows[:limit]
    if pd is not None:
        display(pd.DataFrame(visible))
    else:
        for row in visible:
            print(row)

In [ ]:
OPF_NATIVE_LABELS = {
    "account_number",
    "private_address",
    "private_date",
    "private_email",
    "private_person",
    "private_phone",
    "private_url",
    "secret",
}

SOURCE_TO_OPF_LABEL = {
    "BANCO": None,
    "CBU": "account_number",
    "CORREO_ELECTRONICO": "private_email",
    "CUIJ": "account_number",
    "CUIT_CUIL": "account_number",
    "DIRECCION": "private_address",
    "DNI": "account_number",
    "EDAD": None,
    "ESTUDIOS": None,
    "FECHA": "private_date",
    "IP": "private_url",
    "LINK": "private_url",
    "LOC": "private_address",
    "MARCA_AUTOMOVIL": None,
    "NACIONALIDAD": None,
    "NOMBRE_ARCHIVO": None,
    "NUM_ACTUACION": "account_number",
    "NUM_CAJA_AHORRO": "account_number",
    "NUM_EXPEDIENTE": "account_number",
    "NUM_MATRICULA": "account_number",
    "PATENTE_DOMINIO": "account_number",
    "PER": "private_person",
    "TELEFONO": "private_phone",
    "TEXTO_ANONIMIZAR": "secret",
    "USUARIX": "private_person",
}


def map_label_to_opf(raw_label: str) -> tuple[str | None, str | None]:
    label = raw_label.strip()
    if not label:
        return None, "empty_label"

    mapped = SOURCE_TO_OPF_LABEL.get(label)
    if mapped is None:
        if label in SOURCE_TO_OPF_LABEL:
            return None, "unmapped_label"
        # Allow already OPF-native labels to pass through unchanged.
        if label in OPF_NATIVE_LABELS:
            return label, None
        return None, "unknown_label"

    if mapped not in OPF_NATIVE_LABELS:
        return None, "invalid_mapped_label"
    return mapped, None


def normalize_entity_span(
    text: str, entity: dict[str, Any]
) -> tuple[dict[str, Any] | None, list[str], list[str]]:
    issues: list[str] = []
    warnings: list[str] = []
    raw_label = str(entity.get("label") or "").strip()
    mapped_label, label_issue = map_label_to_opf(raw_label)
    if label_issue == "empty_label":
        issues.append("empty_label")
    elif label_issue is not None:
        warnings.append(f"{label_issue}:{raw_label}")

    if mapped_label is None and not issues:
        return None, issues, warnings

    try:
        start = int(entity.get("start_char"))
        end = int(entity.get("end_char"))
    except (TypeError, ValueError):
        return None, [*issues, "invalid_offsets"], warnings

    if start < 0 or end <= start or end > len(text):
        issues.append("invalid_offsets")

    entity_text = entity.get("text")
    if (
        isinstance(entity_text, str)
        and 0 <= start < end <= len(text)
        and text[start:end] != entity_text
    ):
        issues.append("span_text_mismatch")

    if issues:
        return None, issues, warnings
    return {"category": mapped_label, "start": start, "end": end}, [], warnings


def convert_candidate(
    row: dict[str, Any], row_index: int
) -> tuple[dict[str, Any] | None, list[str], list[str]]:
    text = row.get("text")
    if not isinstance(text, str) or not text:
        return None, ["missing_text"], []

    entities = row.get("final_entities") or []
    if not isinstance(entities, list):
        return None, ["final_entities_not_list"], []

    labels: list[dict[str, Any]] = []
    issues: list[str] = []
    warnings: list[str] = []
    seen_spans: set[tuple[str, int, int]] = set()

    for entity_index, entity in enumerate(entities):
        if not isinstance(entity, dict):
            issues.append(f"entity_not_object:{entity_index}")
            continue
        span, span_issues, span_warnings = normalize_entity_span(text, entity)
        issues.extend(f"{issue}:{entity_index}" for issue in span_issues)
        warnings.extend(f"{warning}:{entity_index}" for warning in span_warnings)
        if span is None:
            continue
        key = (span["category"], span["start"], span["end"])
        if key in seen_spans:
            warnings.append(f"duplicate_span:{entity_index}")
            continue
        seen_spans.add(key)
        labels.append(span)

    if issues:
        return None, issues, warnings

    info = {
        "row_index": row_index,
        "sample_id": row.get("sample_id"),
        "document_id": row.get("document_id"),
        "paragraph_id": row.get("paragraph_id"),
        "final_decision": row.get("final_decision"),
        "source_entity_count": len(entities),
    }
    return {"text": text, "label": labels, "info": info}, [], warnings


def convert_dataset(
    rows: list[dict[str, Any]],
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    converted: list[dict[str, Any]] = []
    rejected: list[dict[str, Any]] = []
    warning_records: list[dict[str, Any]] = []
    issue_counts: Counter[str] = Counter()
    warning_counts: Counter[str] = Counter()

    for row_index, row in enumerate(rows):
        converted_row, issues, warnings = convert_candidate(row, row_index)
        for issue in issues:
            issue_counts[issue.split(":", 1)[0]] += 1
        for warning in warnings:
            warning_counts[warning.split(":", 1)[0]] += 1
        if warnings:
            warning_records.append(
                {
                    "row_index": row_index,
                    "sample_id": row.get("sample_id"),
                    "warnings": warnings,
                }
            )
        if converted_row is None:
            rejected.append(
                {
                    "row_index": row_index,
                    "sample_id": row.get("sample_id"),
                    "issues": issues,
                }
            )
            continue
        converted.append(converted_row)

    report = {
        "input_examples": len(rows),
        "valid_examples": len(converted),
        "rejected_examples": len(rejected),
        "annotated_examples": sum(1 for row in converted if row["label"]),
        "empty_label_examples": sum(1 for row in converted if not row["label"]),
        "total_spans": sum(len(row["label"]) for row in converted),
        "issue_counts": dict(sorted(issue_counts.items())),
        "warning_counts": dict(sorted(warning_counts.items())),
        "rejected_preview": rejected[:25],
        "warning_preview": warning_records[:25],
        "source_to_opf_label": SOURCE_TO_OPF_LABEL,
    }
    return converted, report


In [ ]:
# Smoke test with synthetic data only.
synthetic_rows = [
    {
        "sample_id": "synthetic-1",
        "document_id": "synthetic-doc-1",
        "paragraph_id": "1",
        "text": "Maria Perez vive en Avenida Corrientes 1234.",
        "final_entities": [
            {"label": "PER", "start_char": 0, "end_char": 11, "text": "Maria Perez"},
            {
                "label": "DIRECCION",
                "start_char": 20,
                "end_char": 43,
                "text": "Avenida Corrientes 1234",
            },
        ],
        "final_decision": "synthetic",
    },
    {
        "sample_id": "synthetic-2",
        "document_id": "synthetic-doc-2",
        "paragraph_id": "1",
        "text": "El DNI 12345678 corresponde a Juan Gomez.",
        "final_entities": [
            {"label": "DNI", "start_char": 7, "end_char": 15, "text": "12345678"},
            {"label": "PER", "start_char": 30, "end_char": 40, "text": "Juan Gomez"},
        ],
        "final_decision": "synthetic",
    },
]

synthetic_converted, synthetic_report = convert_dataset(synthetic_rows)
assert synthetic_report["rejected_examples"] == 0, synthetic_report
assert synthetic_converted[0]["label"][0] == {"category": "private_person", "start": 0, "end": 11}
assert synthetic_converted[1]["label"][0] == {"category": "account_number", "start": 7, "end": 15}
print("Synthetic conversion smoke test passed.")
show_table(synthetic_converted)

In [ ]:
if not INPUT_JSONL.exists():
    raise FileNotFoundError(
        "Set AYMURAI_TRAIN_CANDIDATES_JSONL or edit INPUT_JSONL to point to an authorized "
        f"train_candidates.jsonl file. Current value: {INPUT_JSONL}"
    )

candidate_rows = read_jsonl(INPUT_JSONL)
print(f"Loaded {len(candidate_rows):,} candidate rows.")

In [ ]:
# Audit: source labels vs mapped OPF labels coverage.
source_label_counts: Counter[str] = Counter()
mapped_label_counts: Counter[str] = Counter()
unmapped_label_counts: Counter[str] = Counter()

for row in candidate_rows:
    entities = row.get("final_entities") or []
    if not isinstance(entities, list):
        continue
    for entity in entities:
        if not isinstance(entity, dict):
            continue
        raw_label = str(entity.get("label") or "").strip()
        if not raw_label:
            continue
        source_label_counts[raw_label] += 1
        mapped_label, label_issue = map_label_to_opf(raw_label)
        if mapped_label is None:
            unmapped_label_counts[raw_label] += 1
        else:
            mapped_label_counts[mapped_label] += 1

coverage_rows = []
for source_label in sorted(source_label_counts):
    mapped_label, _ = map_label_to_opf(source_label)
    coverage_rows.append(
        {
            "source_label": source_label,
            "source_count": source_label_counts[source_label],
            "mapped_to": mapped_label or "(skipped)",
            "mapped_count": 0 if mapped_label is None else source_label_counts[source_label],
        }
    )

show_table(coverage_rows)

print("\nMapped OPF label totals:")
show_table(
    [
        {"opf_label": label, "count": mapped_label_counts[label]}
        for label in sorted(mapped_label_counts)
    ]
)

if unmapped_label_counts:
    print("\nSkipped/unmapped source labels:")
    show_table(
        [
            {"source_label": label, "count": unmapped_label_counts[label]}
            for label in sorted(unmapped_label_counts)
        ]
    )



In [ ]:
# EDA: raw candidate and label-frequency overview.
raw_label_counts = label_frequency(candidate_rows, field="final_entities")
raw_total_spans = sum(raw_label_counts.values())
raw_annotated_examples = sum(1 for row in candidate_rows if row.get("final_entities"))
raw_empty_label_examples = len(candidate_rows) - raw_annotated_examples

eda_summary = {
    "examples": len(candidate_rows),
    "annotated_examples": raw_annotated_examples,
    "empty_label_examples": raw_empty_label_examples,
    "total_spans": raw_total_spans,
    "unique_labels": len(raw_label_counts),
}
show_table([eda_summary])

raw_label_table = [
    {
        "label": label,
        "count": count,
        "share": count / raw_total_spans if raw_total_spans else 0.0,
    }
    for label, count in raw_label_counts.most_common()
]
show_table(raw_label_table)

if plt is not None and raw_label_table:
    top_labels = raw_label_table[:25]
    fig, ax = plt.subplots(figsize=(10, max(3, len(top_labels) * 0.3)))
    ax.barh(
        [row["label"] for row in reversed(top_labels)],
        [row["count"] for row in reversed(top_labels)],
    )
    ax.set_title("Top label frequencies")
    ax.set_xlabel("Span count")
    ax.set_ylabel("Label")
    fig.tight_layout()
else:
    print("matplotlib is not available or there are no labels to plot.")

In [ ]:
converted_rows, conversion_report = convert_dataset(candidate_rows)
print(json.dumps(conversion_report, indent=2, ensure_ascii=False))

if not converted_rows:
    raise ValueError(
        "No valid OPF examples were produced. Check conversion_report for validation issues."
    )

converted_label_counts: Counter[str] = Counter()
for row in converted_rows:
    for span in row["label"]:
        converted_label_counts[span["category"]] += 1

labels = sorted(converted_label_counts)
if not labels:
    raise ValueError(
        "No labels found in valid examples. OPF custom label-space training requires at least one label."
    )

label_space = {"category_version": CATEGORY_VERSION, "span_class_names": ["O", *labels]}
assert label_space["span_class_names"][0] == "O"
show_table(
    [{"label": label, "count": converted_label_counts[label]} for label in labels]
)

In [ ]:
def split_group_key(example: dict[str, Any], fallback_index: int) -> str:
    info = example.get("info") or {}
    if info.get("document_id"):
        return f"document:{info['document_id']}"
    if info.get("sample_id"):
        return f"sample:{info['sample_id']}"
    return f"row:{fallback_index}"


def deterministic_grouped_split(
    examples: list[dict[str, Any]],
    train_ratio: float,
    validation_ratio: float,
    seed: int,
) -> dict[str, list[dict[str, Any]]]:
    groups: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for index, example in enumerate(examples):
        groups[split_group_key(example, index)].append(example)

    group_keys = sorted(groups)
    random.Random(seed).shuffle(group_keys)
    group_count = len(group_keys)

    if group_count < 3:
        train_keys = set(group_keys)
        validation_keys: set[str] = set()
        test_keys: set[str] = set()
    else:
        train_count = max(1, int(group_count * train_ratio))
        validation_count = max(1, int(group_count * validation_ratio))
        if train_count + validation_count >= group_count:
            validation_count = 1
            train_count = group_count - 2
        train_keys = set(group_keys[:train_count])
        validation_keys = set(group_keys[train_count : train_count + validation_count])
        test_keys = set(group_keys[train_count + validation_count :])

    splits = {"train": [], "validation": [], "test": []}
    for key in group_keys:
        if key in train_keys:
            splits["train"].extend(groups[key])
        elif key in validation_keys:
            splits["validation"].extend(groups[key])
        elif key in test_keys:
            splits["test"].extend(groups[key])
        else:
            raise AssertionError(f"Unassigned group: {key}")
    return splits


splits = deterministic_grouped_split(
    converted_rows, TRAIN_RATIO, VALIDATION_RATIO, SEED
)

split_summary = []
for split_name, split_rows in splits.items():
    split_summary.append(
        {
            "split": split_name,
            "examples": len(split_rows),
            "annotated_examples": sum(1 for row in split_rows if row["label"]),
            "empty_label_examples": sum(1 for row in split_rows if not row["label"]),
            "spans": sum(len(row["label"]) for row in split_rows),
        }
    )
show_table(split_summary)

In [ ]:
# EDA: label frequencies by split after deterministic grouping.
split_label_rows: list[dict[str, Any]] = []
for split_name, split_rows in splits.items():
    counts: Counter[str] = Counter()
    for row in split_rows:
        for span in row["label"]:
            counts[span["category"]] += 1
    for label in labels:
        split_label_rows.append(
            {"split": split_name, "label": label, "count": counts[label]}
        )

if pd is not None:
    split_label_df = pd.DataFrame(split_label_rows)
    display(
        split_label_df.pivot(index="label", columns="split", values="count")
        .fillna(0)
        .astype(int)
    )
else:
    show_table(split_label_rows)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(TRAIN_JSONL, splits["train"])
write_jsonl(VALIDATION_JSONL, splits["validation"])
write_jsonl(TEST_JSONL, splits["test"])
write_json(LABEL_SPACE_JSON, label_space)

conversion_report["output_dir"] = str(OUTPUT_DIR)
conversion_report["split_summary"] = split_summary
conversion_report["label_counts"] = dict(sorted(converted_label_counts.items()))
conversion_report["label_space"] = label_space
write_json(REPORT_JSON, conversion_report)

assert TRAIN_JSONL.exists()
assert VALIDATION_JSONL.exists()
assert TEST_JSONL.exists()
assert LABEL_SPACE_JSON.exists()
assert (
    json.loads(LABEL_SPACE_JSON.read_text(encoding="utf-8"))["span_class_names"][0]
    == "O"
)

for split_path in [TRAIN_JSONL, VALIDATION_JSONL, TEST_JSONL]:
    preview_rows = read_jsonl(split_path)
    for row in preview_rows[:10]:
        assert isinstance(row.get("text"), str)
        assert isinstance(row.get("label"), list)
        for span in row["label"]:
            assert {"category", "start", "end"}.issubset(span)

print("Wrote OPF artifacts:")
for path in [TRAIN_JSONL, VALIDATION_JSONL, TEST_JSONL, LABEL_SPACE_JSON, REPORT_JSON]:
    print(f"- {path}")

In [ ]:
def command_text(args: list[str]) -> str:
    return shlex.join(str(arg) for arg in args)


def run_or_print(
    args: list[str], *, allow_failure: bool = False
) -> subprocess.CompletedProcess[str] | None:
    print(command_text(args))
    if not RUN_OPF:
        return None
    result = subprocess.run(args, text=True, check=False)
    if result.returncode != 0 and not allow_failure:
        raise subprocess.CalledProcessError(result.returncode, args)
    return result


def opf_eval_command(
    dataset_path: Path,
    checkpoint_path: Path,
    eval_mode: str,
    metrics_path: Path,
    predictions_path: Path,
) -> list[str]:
    return [
        sys.executable,
        "-m",
        "opf",
        "eval",
        str(dataset_path),
        "--checkpoint",
        str(checkpoint_path),
        "--device",
        DEVICE,
        "--eval-mode",
        eval_mode,
        "--metrics-out",
        str(metrics_path),
        "--predictions-out",
        str(predictions_path),
    ]

In [ ]:
# Baseline evaluation on the held-out test split.
run_or_print(
    opf_eval_command(
        TEST_JSONL,
        OPF_CHECKPOINT,
        "untyped",
        OUTPUT_DIR / "baseline_untyped_metrics.json",
        OUTPUT_DIR / "baseline_untyped_predictions.jsonl",
    )
)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# Fine-tune OPF with the AymurAI custom label space.
train_command = [
    sys.executable,
    "-m",
    "opf",
    "train",
    str(TRAIN_JSONL),
    "--checkpoint",
    str(OPF_CHECKPOINT),
    "--validation-dataset",
    str(VALIDATION_JSONL),
    "--label-space-json",
    str(LABEL_SPACE_JSON),
    "--output-dir",
    str(FINETUNED_CHECKPOINT_DIR),
    "--overwrite-output",
    "--device",
    DEVICE,
    "--epochs",
    str(EPOCHS),
    "--batch-size",
    str(BATCH_SIZE),
    "--learning-rate",
    str(LEARNING_RATE),
    "--shuffle-seed",
    str(SEED),
]
run_or_print(train_command)

In [ ]:
# Post-fine-tune evaluation on the held-out test split.
run_or_print(
    opf_eval_command(
        TEST_JSONL,
        FINETUNED_CHECKPOINT_DIR,
        "typed",
        OUTPUT_DIR / "post_typed_metrics.json",
        OUTPUT_DIR / "post_typed_predictions.jsonl",
    )
)

# run_or_print(
#     opf_eval_command(
#         TEST_JSONL,
#         FINETUNED_CHECKPOINT_DIR,
#         "untyped",
#         OUTPUT_DIR / "post_untyped_metrics.json",
#         OUTPUT_DIR / "post_untyped_predictions.jsonl",
#     )
# )